# HF 모델 공급망 논문 — 강건성 변형 실험 (Colab 분담분)

노트북(로컬)과 나눠서 돌린다. **이 노트북은 변형 7개**를 맡는다:
`keeptv, noquant, coarse, strictlang, other2unk, strictcom, noov`
(노트북은 `main, tierT0, tierT2, declared`)

- 결과는 구글 드라이브의 `논문_AiModels/04_results/tables/2026-09-25_full_<변형>/` 에 저장되어 노트북과 공유된다.
- 변형 하나에 메모리 약 3GB → 무료 Colab(약 12GB)에서 **2개씩 동시에** 돌린다.
- 예상 시간: 변형당 40~60분 × 7 ÷ 2 ≈ **3~4시간**. 브라우저 탭을 열어 두세요 (닫으면 세션이 끊길 수 있음).
- 끊겨도 괜찮다: 다시 실행하면 **끝난 변형은 건너뛴다**.

순서: 위에서부터 한 칸씩 실행 (Shift+Enter).

In [1]:
# 1) 구글 드라이브 연결
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# 2) 프로젝트 폴더로 이동, 메모리 확인
import os, subprocess
ROOT = '/content/drive/MyDrive/논문_AiModels'
os.chdir(ROOT + '/03_code')
print(os.getcwd())
!free -g | head -2
!nproc

/content/drive/MyDrive/논문_AiModels/03_code
               total        used        free      shared  buff/cache   available
Mem:              12           1           8           0           3          11
2


In [3]:
# 3) 버전 확인: 노트북에서 고친 최신 코드·데이터가 드라이브에 올라왔는지 (하나라도 FAIL 이면 멈추고 잠시 뒤 다시)
import pandas as pd
checks = {
    'runner supports --workers': '--workers' in open('run_all_experiments.py', encoding='utf-8').read(),
    'sim v2 (availability shock)': 'availability' in open('03_simulation/removal_sim.py', encoding='utf-8').read(),
    'sim v2 (union-matched null)': 'def union_matched' in open('03_simulation/removal_sim.py', encoding='utf-8').read(),
    'commercial fix (#4)': '"responsible_ai", "vendor_custom"}' in open('03_simulation/removal_sim.py', encoding='utf-8').read(),
    'tests v2 present': 'test_availability_semantics_and_mirror' in open('03_simulation/test_removal_sim.py', encoding='utf-8').read(),
}
f = pd.read_parquet(ROOT + '/02_data/processed/2026-09-25/model_flags.parquet', columns=['in_T1'])
checks['cleaning fix #1 + reconnection (T1 = 1,631,600)'] = int(f['in_T1'].sum()) == 1631600
e = pd.read_parquet(ROOT + '/02_data/processed/2026-09-25/edges_all.parquet', columns=['parent_resolution', 'temporal_ok'])
checks['parent reconnection (5,820 edges)'] = int(e['parent_resolution'].notna().sum()) == 5820
checks['reconnected edges not time-judged (fix 12:50)'] = int(e.loc[e['parent_resolution'].notna(), 'temporal_ok'].isna().sum()) == 5820
checks['degradation metric'] = 'def degradation' in open('03_simulation/removal_sim.py', encoding='utf-8').read()
checks['arch lower bound'] = os.path.exists(ROOT + '/02_data/processed/2026-09-25/arch.parquet')
for k, v in checks.items():
    print(('OK  ' if v else 'FAIL') + '  ' + k)
assert all(checks.values()), '드라이브 동기화가 아직 안 끝났어요. 몇 분 뒤 이 칸을 다시 실행하세요.'

OK    runner supports --workers
OK    sim v2 (availability shock)
OK    sim v2 (union-matched null)
OK    commercial fix (#4)
OK    tests v2 present
OK    cleaning fix #1 + reconnection (T1 = 1,631,600)
OK    parent reconnection (5,820 edges)
OK    reconnected edges not time-judged (fix 12:50)
OK    degradation metric
OK    arch lower bound


In [4]:
# 4) 필요한 패키지 (Colab 에 대부분 설치돼 있음) + 단위 테스트
!pip -q install -r requirements.txt
!cd 03_simulation && python test_removal_sim.py
!cd 02_graph && python test_rules.py
!python 04_analysis/hypothesis_tests.py --selftest

all tests passed
all rule tests passed
selftest passed: {'H2_availability': {'k_significant': 0, 'k_total': 3, 'supported': False}, 'H2_legal': {'k_significant': 3, 'k_total': 3, 'supported': True}}


In [5]:
# 5) 변형 7개 실행 (2개씩 동시). 몇 시간 걸린다. 진행은 아래 6) 칸에서 확인
!python -u run_all_experiments.py ../02_data/processed/2026-09-25 \
    --only keeptv noquant coarse strictlang other2unk strictcom noov \
    --workers 2 --host colab

[21:30:42] host=colab workers=2 variants=['keeptv', 'noquant', 'coarse', 'strictlang', 'other2unk', 'strictcom', 'noov']
[21:30:42] skip keeptv (done)
[21:30:42] skip noquant (done)
[21:30:42] skip coarse (done)
[21:30:42] skip strictlang (done)
[21:30:42] skip other2unk (done)
[21:30:42] skip strictcom (done)
[21:30:42] skip noov (done)
all done


### 진행 확인 (5번이 도는 동안에는 실행이 안 되므로, 끊겼다가 다시 연결했을 때 사용)

In [6]:
# 6) 진행 상황: 변형별 로그의 마지막 줄과 완료 여부
import glob, os
for name in ['keeptv', 'noquant', 'coarse', 'strictlang', 'other2unk', 'strictcom', 'noov']:
    done = os.path.exists(f'{ROOT}/04_results/tables/2026-09-25_full_{name}/hypothesis_verdicts.json')
    logs = glob.glob(f'{ROOT}/04_results/logs/2026-09-25_{name}_colab.log')
    last = open(logs[0], encoding='utf-8').read().strip().splitlines()[-1][:110] if logs else '(not started)'
    print(f"{'DONE' if done else '....'}  {name:11s} {last}")

DONE  keeptv      [14:39:12] exit 0 (1s)
DONE  noquant     [07:56:16] exit 0 (1s)
DONE  coarse      [14:37:40] exit 0 (1s)
DONE  strictlang  [16:59:54] exit 0 (1s)
DONE  other2unk   [17:02:15] exit 0 (1s)
DONE  strictcom   [19:23:35] exit 0 (3s)
DONE  noov        [19:25:05] exit 0 (1s)


In [ ]:
# 7) 2026-09-27 심사 대응: 대조군 기준 민감도 (tol90, tol99) 동시 실행. 끊겨도 다시 실행하면 이어서 계산
%cd /content/drive/MyDrive/논문_AiModels/03_code
!ls run_logged.py colab_jobs.py
!python colab_jobs.py